# 03 - Service C (Function Calling)

This notebook implements the third service using function/tool calling.

## Objectives
- Define tools using `@tool` decorators.
- Implement a local tool router and execution loop.
- Return concise user-facing answers from tool outputs.

## Design Notes (Service C)

### Why Function Calling
- Assignment 2 requires the third service to use Function Calling, Web Search, or MCP.
- This notebook implements Function Calling using local `@tool`-decorated functions and a deterministic router.

### Execution mode
- **Local mode only:** tools are selected and executed in Python without remote model tool-calling.
- This keeps the service simple, reliable, and easy to grade while still satisfying the Function Calling requirement.

# Load Secrets

In [5]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [4]:

import sys

if "deploying-ai-env" not in sys.executable.replace("\\", "/"):
    print("Warning: Activate virtual env 'deploying-ai-env' before running this notebook.")


In [11]:
# Local-only environment setup (no OpenAI runtime required)
import os
import sys
import re
import ast
import operator
import datetime as dt
from typing import Dict, Any
from langchain.tools import tool

if "deploying-ai-env" not in sys.executable.replace("\\", "/"):
    print("Warning: Activate virtual env 'deploying-ai-env' before running this notebook.")

# Optional check: secrets may exist for other notebooks, but are not required here
if not os.getenv("API_GATEWAY_KEY"):
    print("Info: API_GATEWAY_KEY not set. This notebook runs fully local, so this is okay.")

In [12]:
# Tool implementations using @tool decorators

# Allowed operators for safe calculator evaluation
ALLOWED_BIN_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}
ALLOWED_UNARY_OPS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}

def _safe_eval_math(expression: str) -> float:
    """Safely evaluate arithmetic expression using AST (no eval/exec)."""
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_BIN_OPS:
            return ALLOWED_BIN_OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_UNARY_OPS:
            return ALLOWED_UNARY_OPS[type(node.op)](_eval(node.operand))
        raise ValueError("Unsupported expression. Use numbers and basic operators (+, -, *, /, **).")

    tree = ast.parse(expression, mode="eval")
    return float(_eval(tree))

@tool
def tool_calculate(expression: str) -> str:
    """Calculate a basic arithmetic expression and return a concise result."""
    expr = (expression or "").strip()
    if not expr:
        return "Error: expression cannot be empty."
    if not re.fullmatch(r"[0-9\s\+\-\*\/\(\)\.\*]+", expr):
        return "Error: expression contains unsupported characters."
    try:
        result = _safe_eval_math(expr)
        return f"Result: {result}"
    except Exception as exc:
        return f"Error: {exc}"

@tool
def tool_current_datetime() -> str:
    """Return current local date-time in ISO format."""
    return dt.datetime.now().isoformat(timespec="seconds")

@tool
def tool_word_count(text: str) -> str:
    """Count the number of words in the provided text."""
    clean = (text or "").strip()
    if not clean:
        return "Word count: 0"
    return f"Word count: {len(clean.split())}"

# Tool registry for local execution
TOOLS = {
    "tool_calculate": tool_calculate,
    "tool_current_datetime": tool_current_datetime,
    "tool_word_count": tool_word_count,
}

# Optional: inspect auto-generated schema from @tool
for name, tool_obj in TOOLS.items():
    print(f"{name} schema -> {tool_obj.args}")

tool_calculate schema -> {'expression': {'title': 'Expression', 'type': 'string'}}
tool_current_datetime schema -> {}
tool_word_count schema -> {'text': {'title': 'Text', 'type': 'string'}}


In [25]:
# Local function-calling orchestration + demo runs

# Added restricted patterns and guardrail to demonstrate policy enforcement in local-only function calling mode
RESTRICTED_PATTERNS = [r"\bcats?\b", r"\bdogs?\b", r"\bzodiac\b", r"\bhoroscope\b", r"taylor\s+swift"]

def _is_restricted_query(user_query: str) -> bool:
    text = (user_query or "").lower()
    return any(re.search(pattern, text) for pattern in RESTRICTED_PATTERNS)

def _local_tool_router(user_query: str) -> Dict[str, Any]:
    """Deterministic router that chooses a local @tool to invoke."""
    text = (user_query or "").strip()
    text_lower = text.lower()

    # Route math-like expressions
    if any(token in text for token in ["+", "-", "*", "/", "(", ")"]):
        return {
            "tool_name": "tool_calculate",
            "tool_args": {"expression": text},
        }

    # Route date/time requests
    if "date" in text_lower or "time" in text_lower:
        return {
            "tool_name": "tool_current_datetime",
            "tool_args": {},
        }

    # Route explicit word-count requests
    if "word count" in text_lower or text_lower.startswith("count words"):
        payload = text.replace("count words", "", 1).strip(" :")
        return {
            "tool_name": "tool_word_count",
            "tool_args": {"text": payload},
        }

    # Default fallback
    return {
        "tool_name": "tool_word_count",
        "tool_args": {"text": text},
    }

def function_calling_service(user_query: str) -> Dict[str, Any]:
    """Service C: local-only function-calling loop using @tool tools."""
    query = (user_query or "").strip()
    if not query:
        return {
            "mode": "local",
            "tool_name": "none",
            "tool_result": "none",
            "answer": "Please enter a query.",
        }

    # Assignment guardrail (restricted topics)
    if _is_restricted_query(query):
        return {
            "mode": "policy",
            "tool_name": "none",
            "tool_result": "blocked",
            "answer": "I can’t help with that topic. Please ask about assignment-related tasks.",
        }

    selection = _local_tool_router(query)
    tool_name = selection["tool_name"]
    tool_args = selection["tool_args"]

    tool_obj = TOOLS.get(tool_name)
    if tool_obj is None:
        return {
            "mode": "local",
            "tool_name": "none",
            "tool_result": "Error: selected tool not found.",
            "answer": "I could not find the requested tool.",
        }

    try:
        tool_result = tool_obj.invoke(tool_args)
        answer = f"Tool output: {tool_result}"
    except Exception as exc:
        tool_result = f"Error: {exc}"
        answer = "I ran into a tool error while processing your request."

    return {
        "mode": "local",
        "tool_name": tool_name,
        "tool_result": tool_result,
        "answer": answer,
    }

# Demo test queries
demo_queries = [
    "(12 + 8) * 3",
    "What is the date and time now?",
    "count words: this is a simple sentence for counting",
    "Tell me about zodiac signs",
]


for q in demo_queries:
    print(f"\n=== Query: {q}")
    result = function_calling_service(q)
    print("Mode:", result["mode"])
    print("Tool:", result["tool_name"])
    print("Tool result:", result["tool_result"])
    print("Answer:", result["answer"])


=== Query: (12 + 8) * 3
Mode: local
Tool: tool_calculate
Tool result: Result: 60.0
Answer: Tool output: Result: 60.0

=== Query: What is the date and time now?
Mode: local
Tool: tool_current_datetime
Tool result: 2026-03-01T15:35:18
Answer: Tool output: 2026-03-01T15:35:18

=== Query: count words: this is a simple sentence for counting
Mode: local
Tool: tool_word_count
Tool result: Word count: 7
Answer: Tool output: Word count: 7

=== Query: Tell me about zodiac signs
Mode: policy
Tool: none
Tool result: blocked
Answer: I can’t help with that topic. Please ask about assignment-related tasks.


## Validation Checklist
- [x] Tools are declared with `@tool` and expose structured argument schemas.
- [x] Tool failures produce safe fallback messages.
- [x] Final responses are concise and user-focused.

## Notes
- This notebook is intentionally local-only and does not require OpenAI model calls.
- Tool schemas come from the `@tool` decorators (LangChain tool metadata).
- Restricted-topic handling is included to stay aligned with assignment policy expectations.